# Linear Regression: Ridge and Lasso

**Goal:** Fit linear models to log price. Ridge and Lasso both penalize large coefficients. Ridge shrinks all of them, and Lasso can set some to exactly zero. Their coefficients are the easiest way to explain what drives price.

**Setup:** Same cleaned data, 80/20 split (`random_state=42`), and 5-fold CV as the other notebooks. Numeric features are standardized, so their coefficients are comparable. Categorical features are one-hot encoded.



In [12]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split, GridSearchCV, KFold
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

# Load the cleaned data saved from the KNN notebook's cleaning steps
df = pd.read_csv('../data/cardekho_clean.csv')
df['log_price'] = np.log(df['selling_price'])

num_cols = ['vehicle_age', 'km_driven', 'mileage', 'engine', 'max_power', 'seats']
cat_cols = ['brand', 'model', 'seller_type', 'fuel_type', 'transmission_type']
X = df[num_cols + cat_cols]
y = df['log_price']

# Same split and CV as the KNN notebook so results are comparable
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
cv = KFold(n_splits=5, shuffle=True, random_state=42)

LUXURY = ['Land Rover', 'Jaguar', 'Mercedes-Benz', 'Audi', 'BMW']

def evaluate(pred_log, name):
    rmse = np.sqrt(mean_squared_error(y_test, pred_log))
    r2 = r2_score(y_test, pred_log)
    mae = mean_absolute_error(np.exp(y_test), np.exp(pred_log))
    print(f"{name}: test RMSE (log) {rmse:.4f} | R2 {r2:.4f} | MAE Rs {mae:,.0f}")
    return {'model': name, 'test_rmse_log': round(rmse, 4), 'test_r2': round(r2, 4), 'test_mae_rs': round(mae)}

def segment_report(pred_log):
    d = pd.DataFrame({
        'actual': np.exp(y_test.values),
        'pred': np.exp(pred_log),
        'brand': X_test['brand'].values,
    })
    d['pct_error'] = np.abs(d['pred'] - d['actual']) / d['actual']
    d['segment'] = pd.qcut(d['actual'], 4, labels=['Q1 cheapest', 'Q2', 'Q3', 'Q4 priciest'])
    return d

def print_segments(pred_log):
    d = segment_report(pred_log)
    print(d.groupby('segment', observed=True)['pct_error'].mean().round(3))
    print(d[d['brand'].isin(LUXURY)].groupby('brand')['pct_error'].agg(['size', 'mean']).round(3))

def original_feature(name):
    # Maps one-hot names like 'cat__brand_Maruti' back to 'brand'
    name = name.split('__', 1)[1]
    for col in cat_cols + num_cols:
        if name == col or name.startswith(col + '_'):
            return col
    return name

print(X_train.shape, X_test.shape)

(12184, 11) (3046, 11)


## 1. Ridge: tune the penalty

`alpha` sets how strongly large coefficients are penalized. Larger values shrink the coefficients more. We choose it by CV on the training set only.

In [13]:
from sklearn.linear_model import Ridge

lin_prep = ColumnTransformer([
    ('num', StandardScaler(), num_cols),
    ('cat', OneHotEncoder(handle_unknown='ignore'), cat_cols),
])
lin_pipe = Pipeline([('prep', lin_prep), ('lin', Ridge())])
grid_lin = GridSearchCV(lin_pipe, {'lin__alpha': [0.01, 0.1, 1, 10, 100]}, cv=cv,
                        scoring='neg_root_mean_squared_error', n_jobs=-1)
grid_lin.fit(X_train, y_train)
print("best params:", grid_lin.best_params_)
print("best CV RMSE (log):", round(-grid_lin.best_score_, 4))

best params: {'lin__alpha': 0.1}
best CV RMSE (log): 0.1858


The best value is `alpha = 0.1`, with CV RMSE 0.1858.

## 2. Ridge: test-set evaluation

The test set is used once, here, after the settings were chosen.

In [14]:
best_lin = grid_lin.best_estimator_
pred_lin = best_lin.predict(X_test)
lin_result = evaluate(pred_lin, 'Ridge')

Ridge: test RMSE (log) 0.1814 | R2 0.9278 | MAE Rs 106,284


Ridge's test RMSE is 0.1814 and R² is 0.9278. Its test RMSE is close to its CV RMSE, so the tuning didn't overfit. Its rupee MAE is ₹106,284.

## 3. Ridge: coefficients

- **Numeric features** are standardized, so each coefficient is the change in log price for a one-standard-deviation increase.
- **Categorical features** are one-hot encoded, so each coefficient is the effect of that brand or model relative to the average. Compare categories with each other, not with zero.

In [15]:
coef_names = best_lin.named_steps['prep'].get_feature_names_out()
coefs = pd.Series(best_lin.named_steps['lin'].coef_, index=coef_names)

numeric_coefs = coefs[[n for n in coef_names if n.startswith('num__')]]
numeric_coefs.index = [n.split('__', 1)[1] for n in numeric_coefs.index]
print("Numeric effects (per 1 SD increase, log price):")
print(numeric_coefs.sort_values().round(3))

cat_coefs = coefs[[n for n in coef_names if n.startswith('cat__')]]
print("\nLargest categorical effects (log price):")
print(cat_coefs.reindex(cat_coefs.abs().sort_values(ascending=False).index).head(15).round(3))

Numeric effects (per 1 SD increase, log price):
vehicle_age   -0.329
km_driven     -0.031
engine        -0.006
mileage        0.008
seats          0.009
max_power      0.127
dtype: float64

Largest categorical effects (log price):
cat__brand_Datsun        -1.004
cat__brand_Maruti        -0.791
cat__brand_Ferrari        0.777
cat__model_GTC4Lusso      0.777
cat__brand_Rolls-Royce    0.730
cat__model_Ghost          0.730
cat__brand_Lexus          0.704
cat__brand_Renault       -0.703
cat__brand_Porsche        0.667
cat__model_Endeavour      0.647
cat__brand_Tata          -0.615
cat__brand_Maserati       0.611
cat__model_KWID          -0.595
cat__brand_Hyundai       -0.567
cat__model_KUV           -0.565
dtype: float64


**Numeric effects**
- **Vehicle age** has the largest effect: each standard deviation of age lowers log price by about 0.33.
- **Max power** has the largest positive effect, about +0.13 per standard deviation.
- **Engine size** is close to zero once max power is in the model. The two are strongly related, so they share the same signal.
- **Km driven** has a small negative effect (-0.03), which matches the weak correlation from the EDA.

**Categorical effects**
- Datsun (-1.00) and Maruti (-0.79) have the lowest prices. Ferrari, Rolls-Royce, Lexus, and Porsche are at the top.
- Some estimates for the most expensive brands come from very few cars, so treat them as rough.

## 4. Ridge: error by price segment and brand

In [16]:
print_segments(pred_lin)

segment
Q1 cheapest    0.189
Q2             0.130
Q3             0.116
Q4 priciest    0.136
Name: pct_error, dtype: float64
               size   mean
brand                     
Audi             33  0.135
BMW              78  0.175
Jaguar           14  0.178
Land Rover       11  0.246
Mercedes-Benz    58  0.233


Ridge is most accurate in the middle of the price range. Its error is highest for the cheapest quarter (18.9%), where condition and accident history matter, and lower for the priciest quarter (13.6%).

For luxury brands, Ridge is worse than KNN on most of them. Audi is 13.5%, BMW 17.5%, Jaguar 17.8%, Mercedes-Benz 23.3%, and Land Rover 24.6%. A linear model can't capture how a luxury car's price depends on its specific model.

## 5. Lasso: tune the penalty

Lasso uses an L1 penalty instead of Ridge's L2 penalty. The L1 penalty can shrink some coefficients to exactly zero, so Lasso also works as feature selection.

The first grid's best value was at its lowest edge (0.0001), so we extended the grid downward. The best value is now α = 0.00001, which sits inside the grid, and α = 0.000001 did worse.

In [17]:
from sklearn.linear_model import Lasso

lasso_prep = ColumnTransformer([
    ('num', StandardScaler(), num_cols),
    ('cat', OneHotEncoder(handle_unknown='ignore'), cat_cols),
])
lasso_pipe = Pipeline([('prep', lasso_prep), ('lasso', Lasso(max_iter=10000, random_state=42))])
lasso_grid = {'lasso__alpha': [1e-6, 1e-5, 0.0001, 0.0005, 0.001, 0.005]}
grid_lasso = GridSearchCV(lasso_pipe, lasso_grid, cv=cv,
                          scoring='neg_root_mean_squared_error', n_jobs=-1)
grid_lasso.fit(X_train, y_train)
print("best params:", grid_lasso.best_params_)
print("best CV RMSE (log):", round(-grid_lasso.best_score_, 4))

/Users/jennyyang/miniforge3/lib/python3.12/site-packages/sklearn/linear_model/_coordinate_descent.py:675: ConvergenceWarning: Objective did not converge. You might want to increase the number of iterations, check the scale of the features or consider increasing regularisation. Duality gap: 9.179e+01, tolerance: 4.610e-01
  model = cd_fast.sparse_enet_coordinate_descent(
/Users/jennyyang/miniforge3/lib/python3.12/site-packages/sklearn/linear_model/_coordinate_descent.py:675: ConvergenceWarning: Objective did not converge. You might want to increase the number of iterations, check the scale of the features or consider increasing regularisation. Duality gap: 7.720e+01, tolerance: 4.596e-01
  model = cd_fast.sparse_enet_coordinate_descent(
/Users/jennyyang/miniforge3/lib/python3.12/site-packages/sklearn/linear_model/_coordinate_descent.py:675: ConvergenceWarning: Objective did not converge. You might want to increase the number of iterations, check the scale of the features or consider inc

best params: {'lasso__alpha': 1e-05}
best CV RMSE (log): 0.1857


The best value is `alpha = 0.00001`, with CV RMSE 0.1857. That's essentially tied with Ridge's 0.1858.

## 6. Lasso: test-set evaluation

In [18]:
best_lasso = grid_lasso.best_estimator_
pred_lasso = best_lasso.predict(X_test)
lasso_result = evaluate(pred_lasso, 'Lasso')

Lasso: test RMSE (log) 0.1817 | R2 0.9276 | MAE Rs 106,732


On the test set, Lasso has RMSE 0.1817, R² 0.9276, and MAE ₹106,732. That's very close to Ridge, and slightly behind on all three.

## 7. Lasso: which features were kept

Coefficients at exactly zero were dropped by the model. Lasso kept 138 of the 168 features, so it dropped 30.

In [19]:
lasso_names = best_lasso.named_steps['prep'].get_feature_names_out()
lasso_coefs = pd.Series(best_lasso.named_steps['lasso'].coef_, index=lasso_names)
kept = lasso_coefs[lasso_coefs != 0]
print(f"kept {len(kept)} of {len(lasso_coefs)} features")

print("\nNumeric effects (per 1 SD increase, log price):")
numeric_lasso = lasso_coefs[[n for n in lasso_names if n.startswith('num__')]]
numeric_lasso.index = [n.split('__', 1)[1] for n in numeric_lasso.index]
print(numeric_lasso.sort_values().round(3))

print("\nLargest kept categorical effects (log price):")
cat_kept = kept[[n for n in kept.index if n.startswith('cat__')]]
print(cat_kept.reindex(cat_kept.abs().sort_values(ascending=False).index).head(15).round(3))

kept 138 of 168 features

Numeric effects (per 1 SD increase, log price):
vehicle_age   -0.329
km_driven     -0.030
engine        -0.004
mileage        0.007
seats          0.010
max_power      0.134
dtype: float64

Largest kept categorical effects (log price):
cat__brand_Ferrari        1.477
cat__brand_Rolls-Royce    1.382
cat__brand_Datsun        -1.098
cat__brand_Lexus          0.874
cat__brand_Maserati       0.874
cat__brand_Porsche        0.874
cat__brand_Bentley        0.863
cat__brand_Renault       -0.732
cat__brand_Land Rover     0.725
cat__brand_Maruti        -0.722
cat__model_Endeavour      0.698
cat__brand_Tata          -0.668
cat__model_Glanza        -0.648
cat__brand_Volkswagen    -0.646
cat__model_Wrangler       0.615
dtype: float64


Numeric and categorical effects for this run are pending. The coefficients printed in the last run belong to the earlier α = 0.0001 model, so they are not shown here.

## 8. Lasso: error by price segment and brand

In [20]:
print_segments(pred_lasso)

segment
Q1 cheapest    0.189
Q2             0.130
Q3             0.115
Q4 priciest    0.137
Name: pct_error, dtype: float64
               size   mean
brand                     
Audi             33  0.137
BMW              78  0.176
Jaguar           14  0.180
Land Rover       11  0.247
Mercedes-Benz    58  0.233


Price-segment and brand errors for this run are pending. The breakdown in the last run belongs to the earlier α = 0.0001 model, so it is not shown here.


**Takeaways**
- **Ridge and Lasso are nearly tied.** Their CV RMSE values match to within 0.0001. Ridge is slightly better on the test set, but the gap is tiny.
- **Lasso dropped 30 features and kept 138.** That reduces the model's complexity with almost no loss in accuracy, which is a useful property if the team wants a sparser model.
- **The numeric story is consistent across both models.** Vehicle age is the strongest negative effect, and max power is the strongest positive one. This agrees with the tree and the forest.
- **Linear models struggle with luxury cars.** Neither linear model matches KNN on luxury brands, so the forest is the stronger candidate for those cars.